# 03 – Naive Bayes (K1, Nhóm 1, issue #13)

Notebook riêng cho Gaussian Naive Bayes (không dùng chung `03_classify` để tránh đụng nhau). Logic nằm trong `src/models.py`: `build_naive_bayes`, `prefix_grid`, `tune`, `choose_threshold`, `run_classification` (dùng chung cho #14, #20).

Thiết lập: `clean` → `add_derived_features` → chia có phân tầng (3 tỉ lệ × 5 seed) → mỗi lượt là một `Pipeline` mới `build_preprocessor` → [PCA 11] → `GaussianNB`. Scaler và PCA nằm trong pipeline nên **chỉ fit trên train** (và trên phần train của từng fold khi dò lưới). Dò `var_smoothing` bằng 5-fold CV phân tầng, tối ưu PR-AUC, `balancing = B0_none`. Metric train/val dùng ngưỡng 0.5; cột `val_opt_*` dùng ngưỡng F1 tối ưu chọn **ngay trên validation** nên lạc quan. Kết quả ghi vào `results/results_naive_bayes.csv` (kịch bản `scenario = K1`).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, learning_curve

from src.config import load_config
from src.data import load_raw, TARGET
from src.preprocess import clean, add_derived_features, make_splits
from src.eda import COLORS, FIG_DIR, save_fig
from src import models as md
from src.metrics import load_results

pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 40)
cfg = load_config()
print('seeds', cfg['seeds'], '| splits', cfg['splits'], '| cv', cfg['cv_folds'], cfg['cv_scoring'], '| derived', cfg['use_derived'])
print('grid', cfg['grids']['naive_bayes'])

seeds [0, 1, 2, 3, 4] | splits {'8:2': 0.2, '7:3': 0.3, '6:4': 0.4} | cv 5 average_precision | derived True
grid {'var_smoothing': [1e-09, 1e-07, 1e-05, 0.001, 0.1, 0.3, 1.0, 3.0, 10.0]}


## 0. Dữ liệu và chia train / validation

In [2]:
df = add_derived_features(clean(load_raw()))
y = df[TARGET]
splits = make_splits(y, cfg['seeds'], cfg['splits'])
print(df.shape, '| tỉ lệ vỡ nợ: %.4f' % y.mean())
tr, va = splits[('7:3', 0)]
print('7:3 seed 0 -> train', len(tr), 'val', len(va), '| giao nhau:', len(set(tr) & set(va)))

EDUCATION = 0 -> 4: 14 mẫu
EDUCATION = 5 -> 4: 280 mẫu
EDUCATION = 6 -> 4: 51 mẫu
MARRIAGE = 0 -> 3: 54 mẫu
Loại bản ghi trùng hoàn toàn: 35 dòng (còn 29965)
(29965, 28) | tỉ lệ vỡ nợ: 0.2213
7:3 seed 0 -> train 20975 val 8990 | giao nhau: 0


## 1. Chạy K1: 3 tỉ lệ × 2 dạng dữ liệu × 5 seed = 30 lượt

In [3]:
grid = md.prefix_grid(cfg['grids']['naive_bayes'])
print(grid)
searches = {}
t0 = time.perf_counter()
runs = md.run_classification('naive_bayes', md.build_naive_bayes, grid, cfg, df, splits,
                             data_forms=('full', 'pca11'), searches=searches, verbose=False)
total = time.perf_counter() - t0
print('số lượt:', len(runs), '| tổng thời gian: %.1f s' % total, '| train_time_s trung bình: %.2f' % runs['train_time_s'].mean())
res = load_results('results_naive_bayes.csv')
nb = res[res['model'] == 'naive_bayes']
print('dòng naive_bayes trong results_naive_bayes.csv:', len(nb), '| trùng khóa:',
      int(nb.duplicated(['model', 'ratio', 'data_form', 'balancing', 'derived', 'seed', 'scenario']).sum()))

{'model__var_smoothing': [1e-09, 1e-07, 1e-05, 0.001, 0.1, 0.3, 1.0, 3.0, 10.0]}


số lượt: 30 | tổng thời gian: 43.3 s | train_time_s trung bình: 1.35
dòng naive_bayes trong results_naive_bayes.csv: 30 | trùng khóa: 0


## 2. Bảng tổng hợp mean ± std qua 5 seed

In [4]:
METRICS = ['pr_auc', 'roc_auc', 'f1', 'recall', 'precision', 'balanced_accuracy', 'brier']
order = [(r, f) for r in cfg['splits'] for f in ('full', 'pca11')]

def mean_std(d, cols, digits=4):
    g = d.groupby(['ratio', 'data_form'])[cols]
    m, s = g.mean(), g.std(ddof=1)
    out = m.copy().astype(object)
    for c in cols:
        out[c] = [f'{a:.{digits}f} ± {b:.{digits}f}' for a, b in zip(m[c], s[c])]
    return out.loc[order]

print('VALIDATION (ngưỡng 0.5)')
display(mean_std(runs.assign(**{m: runs['val'].map(lambda d, m=m: d[m]) for m in METRICS}), METRICS))
print('TRAIN (ngưỡng 0.5)')
display(mean_std(runs.assign(**{m: runs['train'].map(lambda d, m=m: d[m]) for m in METRICS}), METRICS))

VALIDATION (ngưỡng 0.5)


pr_auc          roc_auc               f1           recall        precision balanced_accuracy            brier
ratio data_form                                                                                                                        
8:2   full       0.4957 ± 0.0114  0.7503 ± 0.0100  0.3400 ± 0.0655  0.2353 ± 0.0700  0.6469 ± 0.0230   0.5991 ± 0.0280  0.1556 ± 0.0039
      pca11      0.4945 ± 0.0108  0.7492 ± 0.0091  0.0000 ± 0.0000  0.0000 ± 0.0000  0.0000 ± 0.0000   0.5000 ± 0.0000  0.1656 ± 0.0002
7:3   full       0.4959 ± 0.0080  0.7497 ± 0.0076  0.3425 ± 0.0574  0.2376 ± 0.0645  0.6464 ± 0.0262   0.5998 ± 0.0243  0.1557 ± 0.0048
      pca11      0.4947 ± 0.0074  0.7484 ± 0.0069  0.0000 ± 0.0000  0.0000 ± 0.0000  0.0000 ± 0.0000   0.5000 ± 0.0000  0.1655 ± 0.0001
6:4   full       0.4993 ± 0.0042  0.7518 ± 0.0030  0.3715 ± 0.0716  0.2689 ± 0.0805  0.6423 ± 0.0321   0.6123 ± 0.0305  0.1570 ± 0.0053
      pca11      0.4981 ± 0.0043  0.7494 ± 0.0040  0.0000 ± 0.0000  0.0000 ± 0.0000  0.0000 ± 0.0000   0.5000 ± 0.0000  0.1655 ± 0.0001

TRAIN (ngưỡng 0.5)


pr_auc          roc_auc               f1           recall        precision balanced_accuracy            brier
ratio data_form                                                                                                                        
8:2   full       0.4997 ± 0.0024  0.7507 ± 0.0026  0.3429 ± 0.0592  0.2371 ± 0.0665  0.6560 ± 0.0285   0.6003 ± 0.0251  0.1551 ± 0.0042
      pca11      0.4989 ± 0.0026  0.7497 ± 0.0015  0.0000 ± 0.0000  0.0000 ± 0.0000  0.0000 ± 0.0000   0.5000 ± 0.0000  0.1655 ± 0.0001
7:3   full       0.5004 ± 0.0031  0.7511 ± 0.0035  0.3428 ± 0.0626  0.2374 ± 0.0704  0.6566 ± 0.0284   0.6004 ± 0.0267  0.1550 ± 0.0039
      pca11      0.4995 ± 0.0031  0.7498 ± 0.0023  0.0000 ± 0.0000  0.0000 ± 0.0000  0.0000 ± 0.0000   0.5000 ± 0.0000  0.1655 ± 0.0002
6:4   full       0.4988 ± 0.0020  0.7502 ± 0.0037  0.3697 ± 0.0767  0.2689 ± 0.0872  0.6402 ± 0.0360   0.6119 ± 0.0328  0.1570 ± 0.0049
      pca11      0.4979 ± 0.0024  0.7485 ± 0.0022  0.0000 ± 0.0000  0.0000 ± 0.0000  0.0000 ± 0.0000   0.5000 ± 0.0000  0.1655 ± 0.0001

### var_smoothing được chọn (CV PR-AUC) và ngưỡng F1 tối ưu

In [5]:
runs['var_smoothing'] = runs['params'].map(lambda p: p['var_smoothing'])
dist = runs.pivot_table(index=['ratio', 'data_form'], columns='var_smoothing', values='seed', aggfunc='count', fill_value=0).loc[order]
print('Số seed (trên 5) chọn từng giá trị var_smoothing:')
display(dist)
print('CV PR-AUC tốt nhất (mean ± std) và ngưỡng F1 tối ưu chọn trên val (tham khảo, lạc quan):')
opt = runs.assign(cv_pr_auc=runs['cv_pr_auc'], thr_f1=runs['thr_f1'],
                  opt_f1=runs['val_opt'].map(lambda d: d['f1']), opt_recall=runs['val_opt'].map(lambda d: d['recall']),
                  opt_precision=runs['val_opt'].map(lambda d: d['precision']), f1_at_0_5=runs['val'].map(lambda d: d['f1']))
display(mean_std(opt, ['cv_pr_auc', 'thr_f1', 'f1_at_0_5', 'opt_f1', 'opt_recall', 'opt_precision']))

Số seed (trên 5) chọn từng giá trị var_smoothing:


var_smoothing    1.0  3.0
ratio data_form          
8:2   full         1    4
      pca11        0    5
7:3   full         1    4
      pca11        0    5
6:4   full         2    3
      pca11        0    5

CV PR-AUC tốt nhất (mean ± std) và ngưỡng F1 tối ưu chọn trên val (tham khảo, lạc quan):


cv_pr_auc           thr_f1        f1_at_0_5           opt_f1       opt_recall    opt_precision
ratio data_form                                                                                                      
8:2   full       0.5000 ± 0.0024  0.1159 ± 0.0076  0.3400 ± 0.0655  0.5196 ± 0.0099  0.5816 ± 0.0264  0.4704 ± 0.0163
      pca11      0.4997 ± 0.0027  0.2080 ± 0.0019  0.0000 ± 0.0000  0.5200 ± 0.0080  0.5701 ± 0.0293  0.4790 ± 0.0130
7:3   full       0.5005 ± 0.0032  0.1154 ± 0.0113  0.3425 ± 0.0574  0.5193 ± 0.0089  0.5753 ± 0.0175  0.4737 ± 0.0165
      pca11      0.5001 ± 0.0033  0.2086 ± 0.0015  0.0000 ± 0.0000  0.5212 ± 0.0073  0.5640 ± 0.0253  0.4853 ± 0.0144
6:4   full       0.4990 ± 0.0020  0.1037 ± 0.0201  0.3715 ± 0.0716  0.5191 ± 0.0049  0.5915 ± 0.0242  0.4635 ± 0.0165
      pca11      0.4987 ± 0.0025  0.2085 ± 0.0017  0.0000 ± 0.0000  0.5210 ± 0.0025  0.5688 ± 0.0257  0.4818 ± 0.0171

## 3. Overfit

### 3.1 Khoảng cách train − validation (mean ± std qua 5 seed)

In [6]:
gap = runs.copy()
for m in METRICS:
    gap[m] = gap['train'].map(lambda d, m=m: d[m]) - gap['val'].map(lambda d, m=m: d[m])
display(mean_std(gap, METRICS))

pr_auc           roc_auc                f1            recall         precision balanced_accuracy             brier
ratio data_form                                                                                                                              
8:2   full        0.0040 ± 0.0138   0.0004 ± 0.0113   0.0030 ± 0.0181   0.0018 ± 0.0137   0.0090 ± 0.0241   0.0012 ± 0.0073  -0.0005 ± 0.0026
      pca11       0.0044 ± 0.0134   0.0005 ± 0.0106   0.0000 ± 0.0000   0.0000 ± 0.0000   0.0000 ± 0.0000   0.0000 ± 0.0000  -0.0001 ± 0.0003
7:3   full        0.0046 ± 0.0111   0.0014 ± 0.0104   0.0003 ± 0.0148  -0.0002 ± 0.0128   0.0102 ± 0.0082   0.0006 ± 0.0058  -0.0006 ± 0.0020
      pca11       0.0048 ± 0.0105   0.0014 ± 0.0091   0.0000 ± 0.0000   0.0000 ± 0.0000   0.0000 ± 0.0000   0.0000 ± 0.0000  -0.0000 ± 0.0003
6:4   full       -0.0005 ± 0.0061  -0.0016 ± 0.0058  -0.0018 ± 0.0122  -0.0001 ± 0.0121  -0.0021 ± 0.0058  -0.0004 ± 0.0047  -0.0000 ± 0.0008
      pca11      -0.0002 ± 0.0067  -0.0009 ± 0.0061   0.0000 ± 0.0000   0.0000 ± 0.0000   0.0000 ± 0.0000   0.0000 ± 0.0000   0.0000 ± 0.0002

### 3.2 Validation curve theo var_smoothing (7:3, seed 0, CV 5-fold PR-AUC)

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
vc = []
for ax, form in zip(axes, ('full', 'pca11')):
    cvr = searches[('7:3', form, 0)].cv_results_
    x = np.array([p['model__var_smoothing'] for p in cvr['params']])
    for key, lab, col in (('train', 'train (trong fold)', COLORS[0]), ('test', 'validation (trong fold)', COLORS[1])):
        m, s = cvr[f'mean_{key}_score'], cvr[f'std_{key}_score']
        ax.plot(x, m, marker='o', color=col, label=lab)
        ax.fill_between(x, m - s, m + s, color=col, alpha=0.15)
    ax.set_xscale('log'); ax.set_xlabel('var_smoothing'); ax.set_title(f'7:3 seed 0 – {form}')
    ax.grid(alpha=0.3)
    vc.append(pd.DataFrame({'var_smoothing': x, f'{form}_train': cvr['mean_train_score'], f'{form}_val': cvr['mean_test_score']}).set_index('var_smoothing'))
axes[0].set_ylabel('PR-AUC'); axes[0].legend()
p = save_fig(fig, FIG_DIR / 'nb_validation_curve.png')
print('lưu', p.relative_to(ROOT))
display(pd.concat(vc, axis=1).round(4))

lưu results\figures\nb_validation_curve.png


,full_train,full_val,pca11_train,pca11_val
var_smoothing,,,,
1.000000e-09,0.4993,0.4987,0.4761,0.4731
1.000000e-07,0.4993,0.4987,0.4761,0.4731
1.000000e-05,0.4993,0.4987,0.4761,0.4731
1.000000e-03,0.4993,0.4986,0.4763,0.4734
1.000000e-01,0.5002,0.4993,0.4910,0.4896
3.000000e-01,0.5023,0.5015,0.4971,0.4966
1.000000e+00,0.5051,0.5045,0.5025,0.5027
3.000000e+00,0.5048,0.5045,0.5041,0.5048
1.000000e+01,0.5028,0.5031,0.5013,0.5021


![Validation curve NB](../results/figures/nb_validation_curve.png)

### 3.3 Learning curve (7:3, seed 0, dạng 33 chiều, PR-AUC)

In [8]:
Xy = df.drop(columns=TARGET)
tr, va = splits[('7:3', 0)]
X_tr, y_tr = Xy.loc[tr], y.loc[tr]
best = {k.split('__', 1)[1]: v for k, v in searches[('7:3', 'full', 0)].best_params_.items()}
print('var_smoothing dùng cho learning curve:', best)
pipe = md.build_pipeline(md.build_naive_bayes(best), 'full', cfg, seed=0)
sizes, tr_sc, va_sc = learning_curve(pipe, X_tr, y_tr, train_sizes=np.linspace(0.1, 1.0, 8),
                                     cv=StratifiedKFold(cfg['cv_folds'], shuffle=True, random_state=0),
                                     scoring=cfg['cv_scoring'], n_jobs=-1)
fig, ax = plt.subplots(figsize=(6, 4))
for sc, lab, col in ((tr_sc, 'train', COLORS[0]), (va_sc, 'validation (CV)', COLORS[1])):
    ax.plot(sizes, sc.mean(1), marker='o', color=col, label=lab)
    ax.fill_between(sizes, sc.mean(1) - sc.std(1), sc.mean(1) + sc.std(1), color=col, alpha=0.15)
ax.set_xlabel('số mẫu train'); ax.set_ylabel('PR-AUC'); ax.set_title(f'Learning curve NB (var_smoothing={best["var_smoothing"]:g})')
ax.legend(); ax.grid(alpha=0.3)
p = save_fig(fig, FIG_DIR / 'nb_learning_curve.png')
print('lưu', p.relative_to(ROOT))
display(pd.DataFrame({'n_train': sizes, 'train': tr_sc.mean(1).round(4), 'val_cv': va_sc.mean(1).round(4), 'gap': (tr_sc.mean(1) - va_sc.mean(1)).round(4)}))

var_smoothing dùng cho learning curve: {'var_smoothing': 3.0}


lưu results\figures\nb_learning_curve.png


,n_train,train,val_cv,gap
0,1678,0.4874,0.4974,-0.0100
1,3835,0.5257,0.5045,0.0212
2,5992,0.5237,0.5045,0.0192
3,8150,0.5193,0.5043,0.0150
4,10307,0.5148,0.5048,0.0099
5,12465,0.5106,0.5043,0.0064
6,14622,0.5056,0.5046,0.0010
7,16780,0.5048,0.5045,0.0003


![Learning curve NB](../results/figures/nb_learning_curve.png)

## 4. Vì sao Naive Bayes không có regularization đúng nghĩa

Logistic có hệ số phạt (`C`, L1/L2) và SVM có `C`: các tham số này **thu hẹp không gian hàm** (phạt độ lớn trọng số, hạn chế biên), nên giảm phương sai của mô hình. `GaussianNB` không có hàm mục tiêu để cộng thêm số hạng phạt: nó ước lượng trực tiếp, bằng công thức đóng, tiên nghiệm của lớp và trung bình/phương sai từng đặc trưng theo từng lớp (MLE), không có vòng tối ưu nào.

`var_smoothing` chỉ cộng `var_smoothing × max(phương sai của mọi đặc trưng, tính trên toàn bộ dữ liệu train)` vào **phương sai** của từng (lớp, đặc trưng) (theo tài liệu scikit-learn: "portion of the largest variance of all features that is added to variances for calculation stability"). Mục đích là ổn định số học, tránh chia cho phương sai gần 0. Nó **không phạt tham số** và không co trung bình hay tiên nghiệm lớp. Lượng cộng thêm là `var_smoothing × (phương sai lớn nhất của một đặc trưng trên train)`: với 33 chiều đã scale, phương sai lớn nhất = 1 nên cộng đúng `var_smoothing` (3.0 → +3); với PCA 11, thành phần đầu có phương sai ≈ 9.3 nên cộng gấp ≈ 9 lần (var_smoothing = 3.0 → +27.8). Khi `var_smoothing` nhỏ (≤ 1e-3) nó chỉ là hiệu chỉnh số học (validation curve phẳng). Khi lớn (≥ 0.3), nó **lấn át phương sai thật của từng lớp**: phương sai hai lớp trở nên gần bằng nhau (tỉ số phương sai lớp 1/lớp 0 ở 7:3 seed 0, trung vị qua đặc trưng, dạng 33 chiều: ≈ 1.12 khi var_smoothing = 1e-9, còn ≈ 1.03 khi = 3.0), các số hạng bậc hai bị triệt tiêu và NB gần với mô hình tuyến tính có phương sai chung (kiểu LDA đường chéo); đồng thời mỗi đặc trưng đóng góp tỉ số khả năng nhỏ hơn nên xác suất bị kéo về tiên nghiệm, ít cực đoan hơn. Đây là thứ **gần nhất** NB có với regularization (làm mịn/co về mô hình đơn giản hơn: phương sai bằng nhau, bằng chứng từng cột bị giảm trọng), nhưng vẫn là hiệu chỉnh gián tiếp trên phương sai, không phải hình phạt lên tham số; trung bình và tiên nghiệm lớp vẫn là MLE. Ở đây nó giúp thật (PR-AUC val tăng khi `var_smoothing` tăng đến ≈ 1–3, xem mục 3.2) chủ yếu vì giảm việc cộng dồn bằng chứng của các cột tương quan, không phải vì giảm overfit. Số tham số của NB rất ít (2 × 33 trung bình + 2 × 33 phương sai + 2 tiên nghiệm) so với ~21.000 mẫu train, nên nguy cơ overfit thấp; vấn đề chính của NB là **underfit/thiên lệch** do giả định độc lập có điều kiện (các cột PAY_x, BILL_AMTx tương quan mạnh).

## 5. Calibration nhanh (xác suất dự đoán trên validation, 7:3 seed 0)

In [9]:
rows = []
Xv, yv = Xy.loc[va], y.loc[va]
for form in ('full', 'pca11'):
    est = searches[('7:3', form, 0)].best_estimator_
    pr = est.predict_proba(Xv)[:, 1]
    rows.append({'data_form': form, 'tỉ lệ vỡ nợ thật': yv.mean(), 'xác suất dự đoán TB': pr.mean(),
                 'tỉ lệ pr < 0.01': (pr < 0.01).mean(), 'tỉ lệ pr > 0.99': (pr > 0.99).mean(),
                 'tỉ lệ pr < 0.05 hoặc > 0.95': ((pr < 0.05) | (pr > 0.95)).mean(),
                 'Brier': runs[(runs.ratio == '7:3') & (runs.data_form == form) & (runs.seed == 0)]['val'].iloc[0]['brier']})
display(pd.DataFrame(rows).set_index('data_form').round(4))
# cùng thống kê, gộp mọi lượt 7:3 và 5 seed
agg = []
for form in ('full', 'pca11'):
    prs, ys = [], []
    for s in cfg['seeds']:
        _, v = splits[('7:3', s)]
        prs.append(searches[('7:3', form, s)].best_estimator_.predict_proba(Xy.loc[v])[:, 1]); ys.append(y.loc[v].mean())
    pr = np.concatenate(prs)
    agg.append({'data_form': form, 'tỉ lệ vỡ nợ thật (TB 5 seed)': np.mean(ys), 'xác suất dự đoán TB': pr.mean(),
                'tỉ lệ pr < 0.01': (pr < 0.01).mean(), 'tỉ lệ pr > 0.99': (pr > 0.99).mean()})
print('Gộp 5 seed ở 7:3:')
display(pd.DataFrame(agg).set_index('data_form').round(4))

,tỉ lệ vỡ nợ thật,xác suất dự đoán TB,tỉ lệ pr < 0.01,tỉ lệ pr > 0.99,tỉ lệ pr < 0.05 hoặc > 0.95,Brier
data_form,,,,,,
full,0.2212,0.1397,0.0146,0.0039,0.2683,0.1547
pca11,0.2212,0.2025,0.0000,0.0000,0.0000,0.1656


Gộp 5 seed ở 7:3:


,tỉ lệ vỡ nợ thật (TB 5 seed),xác suất dự đoán TB,tỉ lệ pr < 0.01,tỉ lệ pr > 0.99
data_form,,,,
full,0.2212,0.1433,0.0416,0.0098
pca11,0.2212,0.2025,0.0000,0.0000


## 6. Nhận xét (theo số ở trên)

**Hiệu năng (validation, ngưỡng 0.5).** Dạng 33 chiều: PR-AUC 0.4957 / 0.4959 / 0.4993 và ROC-AUC 0.7503 / 0.7497 / 0.7518 ở tỉ lệ 8:2 / 7:3 / 6:4. Dạng PCA 11: PR-AUC 0.4945 / 0.4947 / 0.4981, ROC-AUC 0.7492 / 0.7484 / 0.7494. Hai dạng nay chênh nhau chỉ ≈ 0.001 PR-AUC và ≈ 0.001–0.002 ROC-AUC, nhỏ hơn nhiều độ lệch chuẩn qua seed (0.004–0.011), nên coi như ngang nhau. Đổi tỉ lệ chia gần như không đổi kết quả (khác nhau trong phạm vi std). So với lần chạy trước (lưới chỉ tới 0.1; 33 chiều 0.4895 / 0.4909 / 0.4954, PCA 11 0.4780 / 0.4808 / 0.4826), mở rộng lưới tăng PR-AUC thêm ≈ 0.004–0.006 (33 chiều, cùng cỡ 1 std) và ≈ 0.014–0.017 (PCA 11, lớn hơn std).

**Ngưỡng 0.5 và F1.** Ở 0.5 xác suất đã bị co về gần tiên nghiệm nên NB dự đoán "vỡ nợ" rất ít: 33 chiều có F1 = 0.340 / 0.343 / 0.372, recall ≈ 0.24–0.27, precision ≈ 0.64–0.65 (std của F1 khá lớn, 0.057–0.072). PCA 11 **không dự đoán lớp 1 nào ở ngưỡng 0.5** trong cả 15 lượt (recall = 0, F1 = 0, balanced accuracy = 0.5), dù PR-AUC/ROC-AUC vẫn ≈ 0.495 / 0.749: xếp hạng tốt nhưng xác suất không bao giờ vượt 0.5, nên F1 ở 0.5 không phản ánh năng lực của mô hình này. Chọn ngưỡng F1 tối ưu ngay trên validation (ngưỡng ≈ 0.10–0.12 với 33 chiều, ≈ 0.21 với PCA 11) đưa cả hai về F1 ≈ 0.52 (recall 0.56–0.59, precision 0.46–0.49); con số này **lạc quan** vì ngưỡng chọn trên chính validation, chỉ để tham khảo.

**var_smoothing.** Lưới đã mở rộng đến 10. Giá trị được chọn (mỗi ô tỉ lệ × dạng có 5 seed): 33 chiều chọn 3.0 ở 11/15 lượt và 1.0 ở 4/15; PCA 11 chọn 3.0 ở 15/15 lượt; **không lượt nào chọn 10.0 (biên trên)** và cũng không lượt nào chọn giá trị ≤ 0.3. Vậy tối ưu **nay nằm trong lưới (interior)**, không còn dính biên như khi lưới dừng ở 0.1. Validation curve (7:3, seed 0) xác nhận đỉnh nằm trong khoảng 1–3: 33 chiều phẳng ở 1e-9…1e-3 (≈ 0.4987), tăng dần 0.4993 (0.1) → 0.5015 (0.3) → 0.5045 (1.0 và 3.0, bằng nhau) → giảm còn 0.5031 (10); PCA 11 phẳng ở 0.4731 cho đến 1e-5, rồi 0.4896 (0.1) → 0.4966 (0.3) → 0.5027 (1.0) → **0.5048 (3.0)** → 0.5021 (10). Với 33 chiều, 1.0 và 3.0 gần như ngang nhau nên việc seed chọn 1.0 hay 3.0 là nhiễu, không phải khác biệt thật; phần tăng so với `var_smoothing` nhỏ (≈ +0.006 PR-AUC với 33 chiều, ≈ +0.032 với PCA 11 trong curve) mới là hiệu ứng thật. PCA 11 hưởng lợi nhiều hơn vì phương sai lớn nhất của thành phần PCA ≈ 9.3 nên cùng `var_smoothing` cộng thêm nhiều hơn (xem mục 4). Lưới còn thô (bước ≈ 3 lần), nên giá trị tốt nhất chính xác trong khoảng 1–3 chưa được dò mịn.

**Overfit.** Chênh lệch train − validation rất nhỏ ở mọi cấu hình: PR-AUC từ −0.0005 đến +0.0048, ROC-AUC trong khoảng −0.0016…+0.0014, đều nhỏ hơn std (0.006–0.014). Ở validation curve, train và validation (trong fold) cũng sát nhau ở mọi `var_smoothing` (chênh ≤ 0.002). Learning curve (33 chiều, 7:3 seed 0; giá trị var_smoothing dùng được in ngay dưới ô code): PR-AUC train giảm từ ≈ 0.526 (3.835 mẫu, khoảng cách +0.021) xuống 0.5048, còn validation CV ≈ 0.5045 ở 16.780 mẫu, khoảng cách thu về 0.0003; thêm dữ liệu không giúp vượt ≈ 0.505. Kết luận: NB **không overfit**, hạn chế là thiên lệch (underfit) do giả định độc lập có điều kiện giữa các cột tương quan mạnh; `var_smoothing` lớn cải thiện xếp hạng chủ yếu bằng cách giảm cộng dồn bằng chứng, không phải bằng cách giảm khoảng cách train/val.

**Calibration (7:3 seed 0 trên validation).** Tỉ lệ vỡ nợ thật 0.2212. Dạng 33 chiều: xác suất dự đoán trung bình 0.1397 (**thấp hơn** thực tế; lần chạy trước 0.2437, hơi cao), chỉ 1,5% xác suất < 0.01 và 0,4% > 0.99 (trước: 46,3% và 11,8%), 26,8% nằm ngoài [0.05, 0.95] (trước 76,0%): `var_smoothing` lớn đã bớt đẩy xác suất về hai cực, NB hết quá tự tin nhưng nay hơi dự đoán thấp. Brier 0.1547 **thấp hơn** mức của bộ dự đoán hằng số bằng tỉ lệ vỡ nợ (0.2212 × 0.7788 ≈ 0.172); lần chạy trước cao hơn (0.1971). Dạng PCA 11: trung bình 0.2025 gần tỉ lệ thật, không xác suất nào ngoài [0.05, 0.95] (0% cho cả < 0.01 và > 0.99), Brier 0.1656 chỉ thấp hơn mức hằng số một chút (lần chạy trước 0.1496): xác suất bị co mạnh về tiên nghiệm (không mẫu nào vượt 0.5), xếp hạng tốt hơn nhưng Brier xấu hơn so với khi `var_smoothing` nhỏ vì xác suất kém sắc. Gộp 5 seed ở 7:3 cho cùng kết luận (33 chiều: trung bình 0.1433, 4,2% < 0.01, 1,0% > 0.99; PCA 11: trung bình 0.2025, 0% và 0%). Muốn dùng xác suất NB làm điểm rủi ro thật nên hiệu chỉnh lại (Platt/isotonic), và ngưỡng nên chọn từ validation chứ không dùng 0.5.